# 05 Data Sources — Practical Source-First Setup

This notebook is a practical, executable lesson. Each step follows **Context → Source Problem → Detect → Fix → Verify → Business Result → Exercise**.

## 1. What belongs in the source layer

A source is where the business data originates. At this stage we do not design fact or dimension tables. We discover and collect what the source actually provides.

In [ ]:
from pathlib import Path
root = Path('source_data')
for p in sorted(root.rglob('*')):
    if p.is_file(): print(p)

## 2. SQLite POS source

The POS database contains transaction-level sales records. It is a source database, not a warehouse.

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect('source_data/sqlite/sales_source.db')
print(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", con))
sales = pd.read_sql_query('SELECT * FROM sales_transactions LIMIT 10', con)
sales

## 3. CSV source

CSV files may represent customer registry, stores or weather. Their structure is discovered rather than assumed.

In [ ]:
customers = pd.read_csv('source_data/csv/customer_registry.csv')
stores = pd.read_csv('source_data/csv/store_locations.csv')
weather = pd.read_csv('source_data/csv/weather_station_daily.csv')
for name,df in [('customers',customers),('stores',stores),('weather',weather)]:
    print(name, df.shape, df.dtypes.to_dict())

## 4. Excel source

The product catalog comes from Excel and may contain multiple sheets in a real system.

In [ ]:
xls = pd.ExcelFile('source_data/excel/product_catalog.xlsx')
print(xls.sheet_names)
products = pd.read_excel('source_data/excel/product_catalog.xlsx', sheet_name='ProductCatalog')
products.head()

## 5. XML source

XML represents hierarchical source data. We parse it without turning it into a dimension yet.

In [ ]:
import xml.etree.ElementTree as ET
root_xml = ET.parse('source_data/xml/supplier_catalog.xml').getroot()
rows=[]
for e in root_xml.findall('supplier'):
    rows.append({c.tag:c.text for c in e})
pd.DataFrame(rows)

## 6. Website source

HTML is an external source. We extract only what is required from the page.

In [ ]:
from bs4 import BeautifulSoup
html = Path('source_data/website/competitor_products.html').read_text()
soup = BeautifulSoup(html,'html.parser')
print(soup.title.get_text(strip=True) if soup.title else 'No title')
print('tables:', len(soup.find_all('table')))

## 7. JSON/API source

The API response is source data. We first inspect the payload and then map the fields.

In [ ]:
import json
obj=json.loads(Path('source_data/api/weather_api_response.json').read_text())
print(obj if isinstance(obj,dict) else obj[:1])

## 8. Reference data source

Country/state and controlled code lists are reference data sources. They are not dimensions at this point.

In [ ]:
countries=pd.read_csv('source_data/reference/countries.csv')
states=pd.read_csv('source_data/reference/states.csv')
print(countries)
print(states)

## 9. Source architecture

The source layer should show the heterogeneous reality before engineering creates standardized layers.

In [ ]:
print('SQLite POS -> transactions')
print('CSV -> customer/store/weather')
print('Excel -> product catalog')
print('XML -> supplier catalog')
print('HTML -> competitor data')
print('JSON API -> weather')
print('Parquet/Arrow -> historical/catalog examples')
print('Sensor -> freezer measurements')

## Exercises

1. Run every code cell and inspect the output.
2. Modify at least one input value to create a data problem.
3. Detect the problem with code.
4. Fix it without hiding the original record.
5. Verify the result and explain the business impact.